## Load clean train and test data

In [ ]:
import pandas as pd

train_clean_dummies = pd.read_csv('train_test_clean/train_clean_dummies.csv')
train_clean_no_dummies = pd.read_csv('train_test_clean/train_clean_no_dummies.csv')

test_clean_dummies = pd.read_csv('train_test_clean/test_clean_dummies.csv')
test_clean_no_dummies = pd.read_csv('train_test_clean/test_clean_no_dummies.csv')

## Get time series split for training data, dummies / no dummies

In [ ]:
from scripts.time_series_split import get_moving_windows_split_train, print_folds_shape

target_years = [2015, 2016, 2017, 2018]

train_dummies = get_moving_windows_split_train(train_clean_dummies, target_years, dummies='Y')
train = get_moving_windows_split_train(train_clean_no_dummies, target_years, dummies='N')

#print_folds_shape(train_dummies)
#print_folds_shape(train)

## Preprocess training data for EBM

In [ ]:
from scripts.preprocess_for_models import transform_datatype_for_ebm, apply_preprocessing_to_folds

train_ebm = apply_preprocessing_to_folds(train, transform_datatype_for_ebm)


## Load folds with trained EBMs

In [ ]:
import joblib

folds_ebm = joblib.load("models/folds_ebm.joblib")

## Fit and evaluate LR for range of L1 regularization

In [ ]:
from scripts.l1_regularization_for_lr import fit_lr_parameter_range, get_number_nonzero_weights, plot_dual_axis_nonzero
from scripts.fit_evaluate_models import dataset_info

# note: values below 1e-5 yield no learning
parameter_range = [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1, 10, 1e2, 1e3, 1e4, 1e5, 1e10]

weights_parameters_lr, scores_parameters_lr = fit_lr_parameter_range(train_dummies, dataset_info, parameter_range)

scores_parameters_lr_mean = scores_parameters_lr.mean()

scores_parameters_lr_mean.to_csv("results/scores_parameters_lr_mean.csv", index=False)

lr_nr_nonzero_weights = get_number_nonzero_weights(weights_parameters_lr)
lr_nr_nonzero_weights = pd.DataFrame(lr_nr_nonzero_weights)

lr_nr_nonzero_weights.to_csv("results/lr_nr_nonzero_weights.csv", index=False)

plot_dual_axis_nonzero(parameter_range, scores_parameters_lr_mean, lr_nr_nonzero_weights)

## Fit and evaluate EBMs for range of L1 regularization

In [ ]:
from scripts.ebm_interpretability import fit_evaluate_ebms_l1, plot_ebm_results_l1
import numpy as np

#alphas = [1e1, 1e2, 1e3, 1e4, 1e5]

alphas = np.logspace(1, 5, 13)
print(alphas)

EBM_params = {'interactions': 0.5, 'outer_bags': 9, 'learning_rate': 0.0014, 'min_samples_leaf': 2, 'max_leaves': 3}

fold_0_ebm = train_ebm[0]

df_results, retained_main_features = fit_evaluate_ebms_l1(alphas=alphas, fold=fold_0_ebm , fixed_params=EBM_params)

df_results.to_csv("results/ebm_l1_regularization_results.csv", index=False)

plot_ebm_results_l1(df_results, label_rotate=True, step=1)

## Get EBMs for a range of sparsities (retaining important main effects only)

In [ ]:
from scripts.ebm_interpretability import evaluate_top_n_features, count_main_effects, plot_auc_sparse

fold_0_ebm = folds_ebm[0]
top_n_list = [60, 55, 50, 45, 40, 35, 30, 25, 20, 15, 10, 5]

# use this to see the number of main effects of the ebm in the fold
print(count_main_effects(fold_0_ebm['model']))

EBM_params = {'interactions': 0.5, 'outer_bags': 9, 'learning_rate': 0.0014, 'min_samples_leaf': 2, 'max_leaves': 3}

auc_dict, main_df = evaluate_top_n_features(fold_0_ebm, top_n_list, params=EBM_params)

auc_sparse = pd.DataFrame.from_dict(auc_dict, orient='index', columns=['AUC'])

auc_sparse.to_csv("results/ebm_sparse_auc.csv")

plot_auc_sparse(auc_sparse)




## Train and evaluate sparse EBMs

In [ ]:
from scripts.ebm_interpretability import get_sparse_ebm
from scripts.fit_evaluate_models import save_folds_with_models
import pandas as pd

folds_sparse_45, auc_sparse_45 = get_sparse_ebm(folds_ebm, 45)
folds_sparse_30, auc_sparse_30 = get_sparse_ebm(folds_ebm, 30)
folds_sparse_15, auc_sparse_15 = get_sparse_ebm(folds_ebm, 15)

scores_ebm_45 = pd.DataFrame(auc_sparse_45, columns=['auc_score'])
scores_ebm_30 = pd.DataFrame(auc_sparse_30, columns=['auc_score'])
scores_ebm_15 = pd.DataFrame(auc_sparse_15, columns=['auc_score'])

save_folds_with_models(folds_sparse_30, 'models/folds_ebm_30.joblib')

## Plot sparse EBM scores with other model scores

In [ ]:
from scripts.fit_evaluate_models import plot_model_scores
import pandas as pd

model_scores = pd.read_csv("results/model_scores.csv")

model_scores.drop(columns=['GB', 'RF'], inplace=True)

model_scores['EBM-45'] = scores_ebm_45

model_scores['EBM-30'] = scores_ebm_30

model_scores['EBM-15'] = scores_ebm_15

years = [2015, 2016, 2017, 2018]

plot_model_scores(model_scores, years)

model_scores.to_csv("results/model_scores_2.csv", index=False)

## Load sparse EBM, inspect feature list

In [ ]:
import joblib
import pandas as pd

folds_sparse_30 = joblib.load("models/folds_ebm_30.joblib")

fold_0 = folds_sparse_30[0]

ebm_sparse = fold_0['model_sparse']

ebm_global = ebm_sparse.explain_global()

feature_list = pd.DataFrame(ebm_global.data()['names'])
pd.set_option('display.max_rows', 100)
print(feature_list)

## Get predictor functions for numerical features from EBM, apply smoothing spline

In [ ]:
from scripts.ebm_interpretability import get_ebm_functions, apply_smoothing_spline

feature_names = ['alter', 'vers_verdienst', 'vermittlungsgrad_asal', 'beitragsmonate_vor_rf', 'beschaeftigungsgrad_vorher']

functions_dict = get_ebm_functions(ebm_sparse, feature_names)

functions_dict_smooth = apply_smoothing_spline(functions_dict, function_label='alter', parameter=10)

functions_dict_smooth = apply_smoothing_spline(functions_dict, function_label='vers_verdienst', parameter=1000000)

functions_dict_smooth = apply_smoothing_spline(functions_dict, function_label='beschaeftigungsgrad_vorher', parameter=10000000)

functions_dict_smooth = apply_smoothing_spline(functions_dict, function_label='vermittlungsgrad_asal', parameter=10000)

functions_dict_smooth = apply_smoothing_spline(functions_dict, function_label='beitragsmonate_vor_rf', parameter=1)




## Get EBMs with smoothed features (uniformly across folds)

In [ ]:
from scripts.ebm_interpretability import get_smooth_ebm

# the following values have been determend by modifying the parameters in the previous cell 
smoothing_parameters = {'alter': 10.0, 'vers_verdienst': 1000000.0,  'vermittlungsgrad_asal': 10000.0, 'beitragsmonate_vor_rf': 1.0}
# 'beschaeftigungsgrad_vorher': 10000000.0,

feature_names = ['alter', 'vers_verdienst', 'vermittlungsgrad_asal', 'beitragsmonate_vor_rf']
# , 'beschaeftigungsgrad_vorher'

folds_smoothed, auc_smoothed = get_smooth_ebm(folds_sparse_30, feature_names, smoothing_parameters)

print(auc_smoothed)

## Plot sparse smooth EBM scores with other models

In [ ]:
from scripts.fit_evaluate_models import plot_model_scores

model_scores = pd.read_csv("results/model_scores_2.csv")

model_scores.drop(columns=['EBM-45', 'EBM-15'], inplace=True)

scores_ebm_30_smooth = pd.DataFrame(auc_smoothed, columns=['auc_score'])

model_scores['EBM-30-SM'] = scores_ebm_30_smooth

plot_model_scores(model_scores)

model_scores.to_csv("results/model_scores_3.csv", index=False)

## Edit EBMs using GAMChanger (exploratory)

In [ ]:
# sample code from homepage: https://github.com/interpretml/gam-changer

# Note: this was not used because storing changes with GAMChanger interface did not work properly...

import joblib
import gamchanger as gc

folds_sparse_30 = joblib.load("models/folds_ebm_30.joblib")

fold_0 = folds_sparse_30[0]

ebm_sparse = fold_0['model_sparse']

x_test = fold_0['holdout_X']
y_test = fold_0['holdout_Y']

x_sample = x_test.sample(n=5000, random_state=42)
y_sample = y_test.loc[x_sample.index]

# Extract model weights
model_data = gc.get_model_data(ebm_sparse)

# Generate sample data
sample_data = gc.get_sample_data(ebm_sparse, x_test, y_test)

# Load GAM Changer with the model and sample data
gc.visualize(ebm_sparse, x_sample, y_sample)